# 📧 Lab 6: Spam Email Detector

**Challenge**: Build a classifier that detects spam emails from their text content.

**What you'll use**: pandas, scikit-learn, TF-IDF, Naive Bayes, text processing

---
### The Story
Every email service needs a spam filter. You'll build one! The key challenge: computers don't understand words — you need to convert text into numbers first. We'll use TF-IDF (Term Frequency-Inverse Document Frequency) to do this.

In [ ]:
!pip install pandas scikit-learn matplotlib -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('✅ Ready!')

In [ ]:
# Create a realistic spam dataset
spam_emails = [
    "WINNER! You have won $1,000,000! Click here to claim your prize NOW!",
    "FREE MONEY! Limited time offer! Act now! Call 1-800-SCAM!",
    "Congratulations! You've been selected for a special offer. Buy now!",
    "URGENT: Your account will be suspended. Verify your details immediately!",
    "Make $5000 per week working from home! No experience needed!",
    "You have been pre-approved for a $50,000 loan! Apply now!",
    "Hot singles in your area! Click here to meet them tonight!",
    "Buy cheap medications online! No prescription needed! 90% off!",
    "Your PayPal account has been limited. Click here to restore access!",
    "Earn money fast! Join our network marketing team today!",
    "FREE iPhone 15! You are our lucky winner! Claim before midnight!",
    "ALERT: Suspicious activity on your bank account. Verify now!",
    "Lose 30 pounds in 30 days! Miracle weight loss pill! Order now!",
    "Investment opportunity! 500% returns guaranteed! Limited spots!",
    "Your computer has a virus! Download our FREE antivirus now!",
] * 4  # Repeat to get more samples

ham_emails = [
    "Hi, can we schedule a meeting for tomorrow at 3pm?",
    "Please find attached the report you requested last week.",
    "The project deadline has been moved to Friday. Please update your tasks.",
    "Thanks for your help with the presentation. It went really well!",
    "Reminder: Team lunch is at noon today in the conference room.",
    "I've reviewed your code and left some comments on GitHub.",
    "Can you send me the latest version of the budget spreadsheet?",
    "The client meeting went well. They approved the proposal!",
    "Happy birthday! Hope you have a wonderful day.",
    "I'll be working from home tomorrow. Available on Slack.",
    "The new library version fixes the bug we discussed. Please update.",
    "Your order has been shipped. Expected delivery: 3-5 business days.",
    "Great work on the presentation! The team was impressed.",
    "Can we reschedule our 1:1 to Thursday? I have a conflict.",
    "The quarterly results are in. Revenue is up 15% from last quarter!",
] * 4

emails = spam_emails + ham_emails
labels = [1] * len(spam_emails) + [0] * len(ham_emails)  # 1=spam, 0=ham

df = pd.DataFrame({'email': emails, 'label': labels, 'type': ['spam']*len(spam_emails) + ['ham']*len(ham_emails)})
print(f'Total emails: {len(df)}')
print(f'Spam: {sum(labels)}, Ham: {len(labels)-sum(labels)}')

In [ ]:
# What words appear most in spam vs ham?
from collections import Counter
import re

def get_top_words(texts, n=10):
    words = ' '.join(texts).lower()
    words = re.findall(r'\b[a-z]{3,}\b', words)
    return Counter(words).most_common(n)

spam_words = get_top_words(df[df['label']==1]['email'])
ham_words = get_top_words(df[df['label']==0]['email'])

print('Top spam words:', [w for w,c in spam_words])
print('Top ham words:', [w for w,c in ham_words])

In [ ]:
# Split data
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['email'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

print(f'Train: {len(X_train_text)}, Test: {len(X_test_text)}')

In [ ]:
# YOUR TURN: Convert text to numbers using TF-IDF
# TF-IDF gives higher scores to words that are common in one email but rare overall
# Hint: TfidfVectorizer(max_features=500, stop_words='english')

vectorizer = ___  # ← Create a TfidfVectorizer

# Fit on training text and transform both train and test
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print(f'Feature matrix shape: {X_train.shape}')
print(f'Each email is now represented as {X_train.shape[1]} numbers!')

In [ ]:
# YOUR TURN: Train a Naive Bayes classifier
# Naive Bayes is perfect for text classification!
# Hint: MultinomialNB()

nb_model = ___  # ← Create MultinomialNB
___  # ← Fit on X_train, y_train

nb_preds = nb_model.predict(X_test)
nb_acc = accuracy_score(y_test, nb_preds)
print(f'Naive Bayes Accuracy: {nb_acc:.1%}')
print(classification_report(y_test, nb_preds, target_names=['Ham', 'Spam']))

In [ ]:
# Test your spam detector on new emails!
test_emails = [
    "CONGRATULATIONS! You won a FREE vacation! Click now!",
    "Hi, can we meet tomorrow to discuss the project?",
    "URGENT: Claim your $500 gift card before it expires!",
    "The meeting notes are attached. Please review.",
]

test_vectors = vectorizer.transform(test_emails)
predictions = nb_model.predict(test_vectors)

for email, pred in zip(test_emails, predictions):
    label = '🚫 SPAM' if pred == 1 else '✅ HAM'
    print(f'{label}: {email[:60]}...')

In [ ]:
# ✅ TEST CELL
assert hasattr(vectorizer, 'transform'), 'vectorizer must be fitted'
assert hasattr(nb_model, 'predict'), 'nb_model must be trained'
assert nb_acc >= 0.85, f'Accuracy should be >= 85%, got {nb_acc:.1%}'
assert X_train.shape[1] <= 500, 'TfidfVectorizer should have max_features=500'
print('🎉 All tests passed!')
print(f'Your spam detector accuracy: {nb_acc:.1%}')